# Gov Collect Power Platform

Two flags in this notebook carry most of the product's Power Platform value,
and both are documented Microsoft constraints, not opinions:

* `is_customizable = false` on predefined roles — **Environment Maker cannot
  be edited**. Planning around editing it is planning to fail.
* `security_group_assignable = false` on Default and Developer environments —
  security groups *cannot* be bound there. Combined with `Basic User` +
  `Environment Maker` being auto-assigned in Default and surviving the opt-out,
  this is the one hole the tool can only contain, never close.

⚠️ **A service principal cannot register itself** as a management app. A human
Power Platform Administrator must run `New-PowerAppManagementApp` once. Until
then this notebook fails at the first call, by design — a silent empty result
would look like "no environments", which is a dangerous lie.

In [ ]:
dry_run = True
lakehouse_name = "governance_lh"
key_vault_uri = ""       # e.g. <your-key-vault>.vault.azure.net/
sp_client_id_secret = "gov-pp-client-id"
sp_secret_secret = "gov-pp-secret"
tenant_id = ""
# Dataverse resources are one request set per environment; cap for a first run.
max_environments = 0

In [ ]:
# --- inlined from collectors/shape_common.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Sequence


def utcnow() -> datetime:
    return datetime.now(timezone.utc)


def new_run_id() -> str:
    return str(uuid.uuid4())


def as_str(value: Any) -> str | None:
    """Normalise an API scalar to a string, preserving a real absence as None.

    Collector tables are all-string on purpose: every plane has its own id
    format, and coercing them into typed columns is how a join silently starts
    returning nothing.
    """
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def as_json(value: Any) -> str | None:
    """Stable JSON for a blob column. Sorted keys so diffs are meaningful."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def stamp(rows: Iterable[dict], run_id: str, scanned_at: datetime | None = None) -> list[dict]:
    """Attach the run provenance every `gov_actual_*` row carries."""
    when = scanned_at or utcnow()
    out = []
    for row in rows:
        enriched = dict(row)
        enriched["run_id"] = run_id
        enriched["scanned_at"] = when
        out.append(enriched)
    return out


class RunLedger:
    """Accumulates what a collector did, for the `gov_runs` row and the app.

    Errors are first-class: a collector that quietly drops an unreadable object
    produces a governance report that is wrong in the most dangerous direction —
    it under-reports access.
    """

    def __init__(self, collector: str, module: str, tier: str) -> None:
        self.run_id = new_run_id()
        self.collector = collector
        self.module = module
        self.tier = tier
        self.started_at = utcnow()
        self.finished_at: datetime | None = None
        self.errors: list[dict[str, str]] = []
        self.counts: dict[str, int] = {}

    def count(self, table: str, n: int) -> None:
        self.counts[table] = self.counts.get(table, 0) + n

    def error(self, scope: str, exc: BaseException | str) -> None:
        self.errors.append(
            {
                "scope": scope,
                "type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
                "message": str(exc),
            }
        )

    def finish(self) -> dict:
        self.finished_at = utcnow()
        return {
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "n_objects": sum(self.counts.values()),
            "n_errors": len(self.errors),
            "error_json": as_json(self.errors) if self.errors else None,
            "duration_s": (self.finished_at - self.started_at).total_seconds(),
        }

    def exit_value(self, *, dry_run: bool) -> dict:
        """Actuator-contract-shaped result (PLAN.md §14) for the app to parse."""
        return {
            "ok": True,
            "dry_run": dry_run,
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "counts": dict(self.counts),
            "n_errors": len(self.errors),
            "errors": self.errors[:20],
            "finished_at": (self.finished_at or utcnow()).isoformat(),
        }


def safe_each(
    items: Sequence[Any],
    fn: Callable[[Any], list[dict]],
    ledger: RunLedger,
    scope_of: Callable[[Any], str],
) -> list[dict]:
    """Map `fn` over `items`, recording per-item failures instead of raising."""
    rows: list[dict] = []
    for item in items:
        try:
            rows.extend(fn(item))
        except Exception as exc:  # noqa: BLE001 — a collector must never hard-fail
            ledger.error(scope_of(item), exc)
    return rows

In [ ]:
# --- inlined from collectors/runtime.py (unit-tested offline) ---
from __future__ import annotations

import json
import time
from typing import Any, Callable


class RestError(RuntimeError):
    def __init__(self, status: int, url: str, body: str) -> None:
        super().__init__(f"{status} {url}: {body[:400]}")
        self.status = status
        self.url = url


def fabric_client():
    """A `sempy` REST client for Fabric / Power BI, under the running identity."""
    import sempy.fabric as fabric  # type: ignore

    return fabric.FabricRestClient()


def rest_get(client, path: str, *, retries: int = 4) -> dict[str, Any]:
    """GET with backoff on 429/5xx.

    Admin APIs are rate-limited (25 req/min on some tenant-setting endpoints), and
    a nightly crawl that gives up on the first 429 silently under-reports — which
    is the worst possible failure mode for a governance inventory.
    """
    delay = 2.0
    last: Exception | None = None
    for _ in range(retries):
        response = client.get(path)
        if response.status_code == 200:
            return response.json() if response.text else {}
        if response.status_code in (429, 500, 502, 503, 504):
            retry_after = response.headers.get("Retry-After")
            time.sleep(float(retry_after) if retry_after else delay)
            delay = min(delay * 2, 60)
            last = RestError(response.status_code, path, response.text)
            continue
        raise RestError(response.status_code, path, response.text)
    raise last or RestError(0, path, "exhausted retries")


def graph_token(scope: str = "https://graph.microsoft.com/.default") -> str:
    """Delegated Graph token for the identity the notebook runs as."""
    import notebookutils  # type: ignore

    return notebookutils.credentials.getToken(scope)


def graph_get(token: str, url: str, *, retries: int = 4) -> dict[str, Any]:
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    delay = 2.0
    for _ in range(retries):
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        try:
            with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 500, 502, 503, 504):
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc
    raise RestError(0, url, "exhausted retries")


def graph_call(token: str, method: str, url: str, body: dict | None = None) -> dict[str, Any]:
    """Graph request with a method — the write-capable sibling of `graph_get`.

    Deliberately **not** retried on 5xx: a POST that may have partially applied
    must not be replayed blindly. The actuator's read-before-write makes a
    retry safe only after re-reading, and that is the caller's decision.
    """
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header("Authorization", f"Bearer {token}")
    if data is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as exc:
        raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc


def fabric_call(client, method: str, path: str, body: dict | None = None) -> dict[str, Any]:
    """Fabric REST with a method, through the `sempy` client.

    Same no-retry stance as `graph_call`, for the same reason.
    """
    verb = method.upper()
    if verb == "GET":
        response = client.get(path)
    elif verb == "POST":
        response = client.post(path, json=body or {})
    elif verb == "PATCH":
        response = client.patch(path, json=body or {})
    elif verb == "DELETE":
        response = client.delete(path)
    else:
        raise ValueError(f"unsupported method {method}")

    if response.status_code not in (200, 201, 202, 204):
        raise RestError(response.status_code, path, response.text)
    return response.json() if response.text else {}


def write_table(
    spark,
    lakehouse: str,
    table: str,
    rows: list[dict],
    *,
    dry_run: bool,
    log: Callable[[str, str, str], None],
) -> int:
    """Overwrite one `gov_actual_*` table with this run's rows.

    Overwrite, not append: these tables are a *snapshot of current reality*, and
    the run ledger plus `gov_audit` carry the history. An append-only actual-state
    table is how a drift engine starts comparing against last month.
    """
    if dry_run:
        log(table, "Planned", f"{len(rows)} rows")
        return len(rows)
    if not rows:
        log(table, "Skipped (no permission)", "no rows collected")
        return 0
    try:
        df = spark.createDataFrame(rows)
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            f"{lakehouse}.{table}"
        )
        log(table, "Created", f"{len(rows)} rows")
        return len(rows)
    except Exception as exc:  # noqa: BLE001
        log(table, "Failed", f"{type(exc).__name__}: {exc}")
        return 0


def write_run_row(spark, lakehouse: str, summary: dict, *, dry_run: bool) -> None:
    if dry_run:
        return
    try:
        spark.createDataFrame([summary]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse}.gov_runs")
    except Exception as exc:  # noqa: BLE001
        print(f"gov_runs append failed: {exc}")


def finish(ledger, spark, lakehouse: str, *, dry_run: bool) -> str:
    summary = ledger.finish()
    write_run_row(spark, lakehouse, summary, dry_run=dry_run)
    result = ledger.exit_value(dry_run=dry_run)
    try:
        import notebookutils  # type: ignore

        notebookutils.notebook.exit(json.dumps(result))
    except ImportError:
        print(json.dumps(result, indent=2))
    return json.dumps(result)

In [ ]:
# --- inlined from collectors/shape_pp.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any



#: Predefined Dataverse roles. Microsoft: "You can't edit these roles."
PREDEFINED_ROLES = {
    "Environment Maker",
    "Basic User",
    "System Administrator",
    "System Customizer",
    "Delegate",
    "Environment Admin",
    "Service Reader",
    "Support User",
    "SharePoint custom form maker",
}

#: Environment types that cannot be bound to an Entra security group.
NO_SECURITY_GROUP_TYPES = {"Default", "Developer"}

#: Dataverse tables whose privileges gate agent authoring in Copilot Studio.
AGENT_AUTHORING_TABLES = {"bot", "botcomponent", "conversationtranscript"}


def empty_environments_warning(payload: dict[str, Any]) -> str | None:
    """Return a warning when the admin API reports *no* environments at all.

    Every tenant has at least a Default environment, so an empty list almost
    always means the caller was never registered with
    ``New-PowerAppManagementApp`` (§17.2 A2) — the API answers **200 with an
    empty list**, not 403, so nothing else distinguishes "not allowed to see
    them" from "there are none".

    Recording that as a clean `0 found` would let the Can-Do Explorer answer
    *"nobody can do this in Power Platform"* from an unknown, which is the one
    inference this product must never make (D41).
    """
    if payload.get("value") or []:
        return None
    return (
        "the Power Platform admin API returned 200 with zero environments — "
        "every tenant has at least a Default environment, so this most likely "
        "means the management app was never registered "
        "(New-PowerAppManagementApp). Treating M-PP as under-reporting rather "
        "than reporting an empty tenant."
    )


def shape_environments(payload: dict[str, Any]) -> list[dict]:
    """`GET .../scopes/admin/environments?$expand=properties`."""
    rows: list[dict] = []
    for env in payload.get("value", []) or []:
        props = env.get("properties") or {}
        env_type = as_str((props.get("environmentSku") or props.get("environmentType")))
        governance = props.get("governanceConfiguration") or {}
        protection = as_str(governance.get("protectionLevel"))
        security_group_id = as_str(
            (props.get("linkedEnvironmentMetadata") or {}).get("securityGroupId")
            or props.get("securityGroupId")
        )
        assignable = env_type not in NO_SECURITY_GROUP_TYPES

        rows.append(
            {
                "environment_id": as_str(env.get("name") or env.get("id")),
                "environment_name": as_str(props.get("displayName")),
                "environment_type": env_type,
                "region": as_str(props.get("azureRegion") or props.get("location")),
                "has_dataverse": as_str(bool(props.get("linkedEnvironmentMetadata"))),
                "security_group_id": security_group_id,
                # The governance-critical pair: is a security group *possible*
                # here, and is one actually bound?
                "security_group_assignable": as_str(assignable),
                "security_group_bound": as_str(bool(security_group_id)),
                "is_managed_env": as_str(bool(protection and protection != "Basic")),
                "protection_level": protection,
                "environment_group_id": as_str(props.get("parentEnvironmentGroup", {}).get("id"))
                if isinstance(props.get("parentEnvironmentGroup"), dict)
                else None,
                "created_by": as_str((props.get("createdBy") or {}).get("displayName")),
                "created_at": as_str(props.get("createdTime")),
            }
        )
    return rows


def shape_roles(environment_id: str, payload: dict[str, Any]) -> list[dict]:
    """Dataverse `roles` for one environment."""
    rows: list[dict] = []
    for role in payload.get("value", []) or []:
        name = as_str(role.get("name")) or ""
        rows.append(
            {
                "environment_id": as_str(environment_id),
                "role_id": as_str(role.get("roleid")),
                "role_name": name,
                "is_predefined": as_str(name in PREDEFINED_ROLES),
                # Predefined roles cannot be edited. This is the flag the app
                # uses to refuse to plan a change that Microsoft will reject.
                "is_customizable": as_str(name not in PREDEFINED_ROLES),
                "business_unit_id": as_str(
                    (role.get("_businessunitid_value") or role.get("businessunitid"))
                ),
            }
        )
    return rows


#: Dataverse privilege depth, least → most reach.
DEPTH_BY_CODE = {0: "None", 1: "User", 2: "BusinessUnit", 4: "Parent", 8: "Organization"}

_PRIVILEGE_PREFIXES = (
    ("prvCreate", "Create"),
    ("prvRead", "Read"),
    ("prvWrite", "Write"),
    ("prvDelete", "Delete"),
    ("prvAppendTo", "AppendTo"),
    ("prvAppend", "Append"),
    ("prvAssign", "Assign"),
    ("prvShare", "Share"),
)


def parse_privilege_name(name: str) -> tuple[str | None, str | None]:
    """`prvCreatebot` → (`Create`, `bot`).

    Order matters: `prvAppendTo` must be tested before `prvAppend`, or every
    AppendTo privilege is mis-parsed as an Append on a table called `To…`.
    """
    for prefix, verb in _PRIVILEGE_PREFIXES:
        if name.startswith(prefix):
            return verb, name[len(prefix) :] or None
    return None, None


def shape_role_privileges(
    environment_id: str, role_id: str, payload: dict[str, Any]
) -> list[dict]:
    """Table privileges held by one role, filtered to what governs creation."""
    rows: list[dict] = []
    for privilege in payload.get("value", []) or []:
        raw_name = str(privilege.get("name") or "")
        verb, table = parse_privilege_name(raw_name)
        if not verb or not table:
            continue
        depth_code = privilege.get("depth")
        rows.append(
            {
                "environment_id": as_str(environment_id),
                "role_id": as_str(role_id),
                "privilege_name": raw_name,
                "table_logical_name": table,
                "privilege": verb,
                "depth": DEPTH_BY_CODE.get(depth_code, as_str(depth_code)),
                # The agent-authoring flag: `Create` on `bot` is the supported
                # lever for "who may build a Copilot Studio agent here".
                "gates_agent_authoring": as_str(
                    table in AGENT_AUTHORING_TABLES and verb in {"Create", "Write"}
                ),
            }
        )
    return rows


def shape_role_assignments(environment_id: str, payload: dict[str, Any]) -> list[dict]:
    """`systemuserroles` / `teamroles` → who holds which role.

    Group *teams* are the preferred assignment target: they let an entitlement
    compile onto an Entra group instead of a person.
    """
    rows: list[dict] = []
    for assignment in payload.get("value", []) or []:
        team_type = assignment.get("teamtype")
        is_team = assignment.get("principal_kind") == "Team" or team_type is not None
        rows.append(
            {
                "environment_id": as_str(environment_id),
                "principal_id": as_str(
                    assignment.get("systemuserid") or assignment.get("teamid")
                ),
                "principal_type": "Team" if is_team else "User",
                "principal_name": as_str(
                    assignment.get("fullname")
                    or assignment.get("name")
                    or assignment.get("domainname")
                ),
                "team_type": as_str(assignment.get("teamtype_label") or team_type),
                "azure_group_id": as_str(assignment.get("azureactivedirectoryobjectid")),
                "role_id": as_str(assignment.get("roleid")),
                "role_name": as_str(assignment.get("role_name")),
            }
        )
    return rows


def shape_resources(
    environment_id: str, resource_type: str, payload: dict[str, Any]
) -> list[dict]:
    """Canvas apps / flows / agents in one environment."""
    rows: list[dict] = []
    for resource in payload.get("value", []) or []:
        owner = (
            resource.get("owner")
            or resource.get("createdby")
            or resource.get("_ownerid_value")
        )
        owner_name = owner.get("displayName") if isinstance(owner, dict) else as_str(owner)
        rows.append(
            {
                "environment_id": as_str(environment_id),
                "resource_type": resource_type,
                "resource_id": as_str(
                    resource.get("name") or resource.get("botid") or resource.get("id")
                ),
                "resource_name": as_str(
                    resource.get("displayName")
                    or resource.get("name")
                    or resource.get("schemaname")
                ),
                "owner_name": as_str(owner_name),
                "created_at": as_str(
                    resource.get("createdTime") or resource.get("createdon")
                ),
                "state": as_str(resource.get("statecode") or resource.get("state")),
                "is_orphaned": as_str(not owner_name),
            }
        )
    return rows


def shape_dlp(payload: dict[str, Any]) -> list[dict]:
    """Data (DLP) policies and the environments they cover.

    No licence prerequisite and no Managed Environments requirement — this is
    the primary licence-free lever, especially for the Default environment.
    """
    rows: list[dict] = []
    for policy in payload.get("value", []) or []:
        environments = (policy.get("environments") or {}).get("environments") or []
        env_ids = [as_str(e.get("name") or e.get("id")) for e in environments] or [None]
        default_group = as_str(policy.get("defaultConnectorClassification"))
        groups = policy.get("connectorGroups") or []
        # A custom-connector URL pattern rule is one of the six licence-free
        # levers for hardening the Default environment, so it gets its own flag
        # rather than being buried in the JSON blob.
        blocks_custom = any(
            (g.get("classification") == "Blocked")
            and (g.get("connectors") or g.get("customConnectorUrlPatternsDefinition"))
            for g in groups
            if isinstance(g, dict)
        )
        for env_id in env_ids:
            rows.append(
                {
                    "policy_id": as_str(policy.get("name") or policy.get("policyName")),
                    "policy_name": as_str(policy.get("displayName")),
                    "environment_id": env_id,
                    "scope": as_str(policy.get("environmentType")),
                    "default_connector_group": default_group,
                    "blocks_new_connectors_by_default": as_str(
                        default_group == "Blocked"
                    ),
                    "blocks_custom_connector_urls": as_str(blocks_custom),
                    "connector_groups_json": as_json(groups),
                }
            )
    return rows


#: Tenant-level Power Platform settings the Default-environment posture scores
#: against. Keys are the paths inside the BAP `listtenantsettings` response.
POSTURE_SETTINGS = {
    "disableShareWithEveryone": ("powerPlatform", "powerApps", "disableShareWithEveryone"),
    "disableEnvironmentCreationByNonAdminUsers": (
        "powerPlatform",
        "governance",
        "disableEnvironmentCreationByNonAdminUsers",
    ),
    "disableTrialEnvironmentCreationByNonAdminUsers": (
        "powerPlatform",
        "governance",
        "disableTrialEnvironmentCreationByNonAdminUsers",
    ),
    "disableDeveloperEnvironmentCreationByNonAdminUsers": (
        "powerPlatform",
        "governance",
        "disableDeveloperEnvironmentCreationByNonAdminUsers",
    ),
    "disableCapacityAllocationByEnvironmentAdmins": (
        "powerPlatform",
        "governance",
        "disableCapacityAllocationByEnvironmentAdmins",
    ),
}


def _dig(payload: dict[str, Any], path: tuple[str, ...]) -> Any:
    node: Any = payload
    for key in path:
        if not isinstance(node, dict):
            return None
        node = node.get(key)
    return node


def shape_pp_tenant_settings(payload: dict[str, Any]) -> list[dict]:
    """`POST .../listtenantsettings` → one row per governance-relevant setting.

    Flattened rather than stored as a blob so the posture page can score against
    named settings without re-parsing nested JSON in the browser.
    """
    rows: list[dict] = []
    for name, path in POSTURE_SETTINGS.items():
        value = _dig(payload, path)
        rows.append(
            {
                "setting_name": name,
                "value": as_str(value),
                "is_set": as_str(value is not None),
                "source": "TenantSettings",
                # Same key set as the isolation row below: Spark infers one
                # schema for the whole table, and a row with a missing key is
                # how that inference silently drops a column.
                "detail_json": None,
            }
        )
    return rows


def shape_tenant_isolation(payload: dict[str, Any]) -> list[dict]:
    """`Get-PowerAppTenantIsolationPolicy` → a single posture row.

    Cross-tenant isolation is one of the six licence-free Default-environment
    levers, and it is tenant-wide rather than per-environment — hence its own
    row in the same table instead of a column somewhere.
    """
    properties = payload.get("properties") or payload
    enabled = properties.get("isDisabled")
    # The API expresses this as `isDisabled`, which is easy to invert by
    # accident. Normalise once, here, where it is tested.
    is_enabled = None if enabled is None else (not bool(enabled))
    rules = properties.get("allowedTenants") or []
    return [
        {
            "setting_name": "tenantIsolation",
            "value": as_str(is_enabled),
            "is_set": as_str(enabled is not None),
            "source": "TenantIsolation",
            "detail_json": as_json(rules),
        }
    ]

In [ ]:
import json
import urllib.parse
import urllib.request

steps = []


def log(step, status, detail=""):
    steps.append({"step": step, "status": status, "detail": detail})
    print(f"[{status:>22}] {step}{(' — ' + detail) if detail else ''}")


ledger = RunLedger("Gov Collect PowerPlatform", "pp", "T1")
print(f"run_id={ledger.run_id} dry_run={dry_run}")


def _secret(name):
    """Read a secret from the customer's Key Vault.

    The FULL vault URI is required — the short name fails with
    'Invalid vault uri' (learned in the Data Catalog build).
    """
    import notebookutils  # type: ignore

    return notebookutils.credentials.getSecret(key_vault_uri, name)


def sp_token(resource):
    """Client-credentials token for a service principal.

    The secret is read inside this notebook and never leaves it. It is never in
    the SPA bundle, which Fabric static hosting serves anonymously.
    """
    body = urllib.parse.urlencode(
        {
            "grant_type": "client_credentials",
            "client_id": _secret(sp_client_id_secret),
            "client_secret": _secret(sp_secret_secret),
            "scope": f"{resource}/.default",
        }
    ).encode()
    request = urllib.request.Request(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data=body,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
        return json.loads(response.read().decode())["access_token"]


def api_get(token, url):
    request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
    with urllib.request.urlopen(request) as response:  # noqa: S310
        return json.loads(response.read().decode())


BAP = "https://api.bap.microsoft.com"
BAP_VERSION = "api-version=2020-10-01"

## Environments

In [ ]:
env_rows = []
raw_envs = []
try:
    bap_token = sp_token("https://service.powerapps.com")
    payload = api_get(
        bap_token,
        f"{BAP}/providers/Microsoft.BusinessAppPlatform/scopes/admin/environments"
        f"?$expand=properties&{BAP_VERSION}",
    )
    raw_envs = payload.get("value", []) or []
    env_rows = shape_environments(payload)
    ledger.count("gov_actual_pp_environments", len(env_rows))
    empty_warning = empty_environments_warning(payload)
    if empty_warning:
        ledger.error("environments", empty_warning)
        log("environments", "Unknown", "0 returned — recorded as under-reporting")
    else:
        log("environments", "Created", f"{len(env_rows)} found")
except Exception as exc:  # noqa: BLE001
    ledger.error("environments", exc)
    log("environments", "Failed", f"{type(exc).__name__}: {exc}")

if max_environments and len(raw_envs) > max_environments:
    raw_envs = raw_envs[:max_environments]
    ledger.error("environments", f"capped at {max_environments}")

## Dataverse: roles, privileges, assignments, resources

Only environments that actually have a Dataverse database have security roles.
Everything below is read through the **unlicensed application user**.

In [ ]:
role_rows = []
privilege_rows = []
assignment_rows = []
resource_rows = []

by_id = {r["environment_id"]: r for r in env_rows}

for env in raw_envs:
    env_id = env.get("name") or env.get("id")
    shaped = by_id.get(env_id, {})
    if shaped.get("has_dataverse") != "true":
        continue
    instance_url = (
        ((env.get("properties") or {}).get("linkedEnvironmentMetadata") or {})
        .get("instanceApiUrl")
    )
    if not instance_url:
        ledger.error(f"dataverse:{env_id}", "no instanceApiUrl")
        continue

    try:
        dv_token = sp_token(instance_url)
        api = f"{instance_url}/api/data/v9.2"

        roles_payload = api_get(dv_token, f"{api}/roles?$select=roleid,name")
        env_roles = shape_roles(env_id, roles_payload)
        role_rows.extend(env_roles)

        # Privileges only for roles that could gate agent authoring or that we
        # manage; pulling every privilege of every role is a very large read.
        for role in env_roles:
            if role["is_customizable"] != "true" and role["role_name"] != "Environment Maker":
                continue
            try:
                privileges = api_get(
                    dv_token,
                    f"{api}/roles({role['role_id']})/roleprivileges_association"
                    "?$select=name,privilegeid",
                )
                privilege_rows.extend(
                    shape_role_privileges(env_id, role["role_id"], privileges)
                )
            except Exception as exc:  # noqa: BLE001
                ledger.error(f"privileges:{env_id}:{role['role_name']}", exc)

        for entity, kind in (("systemusers", "User"), ("teams", "Team")):
            try:
                payload = api_get(dv_token, f"{api}/{entity}?$top=5000")
                for row in payload.get("value", []) or []:
                    row["principal_kind"] = kind
                assignment_rows.extend(shape_role_assignments(env_id, payload))
            except Exception as exc:  # noqa: BLE001
                ledger.error(f"assignments:{env_id}:{entity}", exc)

        for entity, resource_type in (
            ("bots", "Agent"),
            ("workflows", "Flow"),
            ("canvasapps", "CanvasApp"),
        ):
            try:
                payload = api_get(dv_token, f"{api}/{entity}?$top=5000")
                resource_rows.extend(shape_resources(env_id, resource_type, payload))
            except Exception as exc:  # noqa: BLE001
                ledger.error(f"resources:{env_id}:{entity}", exc)

    except Exception as exc:  # noqa: BLE001
        ledger.error(f"dataverse:{shaped.get('environment_name') or env_id}", exc)

ledger.count("gov_actual_pp_roles", len(role_rows))
ledger.count("gov_actual_pp_role_privileges", len(privilege_rows))
ledger.count("gov_actual_pp_role_assignments", len(assignment_rows))
ledger.count("gov_actual_pp_resources", len(resource_rows))
log("dataverse", "Created", f"{len(role_rows)} roles, {len(resource_rows)} resources")

## Data policies

No licence prerequisite and no Managed Environments requirement — the primary
licence-free lever, especially for the Default environment.

In [ ]:
dlp_rows = []
try:
    payload = api_get(
        bap_token,
        f"{BAP}/providers/PowerPlatform.Governance/v1/policies?{BAP_VERSION}",
    )
    dlp_rows = shape_dlp(payload)
    ledger.count("gov_actual_pp_dlp", len(dlp_rows))
    log("data policies", "Created", f"{len(dlp_rows)} policy/environment rows")
except Exception as exc:  # noqa: BLE001
    ledger.error("dlp", exc)
    log("data policies", "Skipped (no permission)", str(exc))

## Tenant settings and tenant isolation

These are the remaining licence-free levers the Default-environment posture
scores against. Default cannot be bound to a security group, so this is what
containment actually looks like there.

In [ ]:
posture_rows = []
try:
    payload = api_get(
        bap_token,
        f"{BAP}/providers/Microsoft.BusinessAppPlatform/listtenantsettings?{BAP_VERSION}",
    )
    posture_rows.extend(shape_pp_tenant_settings(payload))
except Exception as exc:  # noqa: BLE001
    ledger.error("ppTenantSettings", exc)

try:
    payload = api_get(
        bap_token,
        f"{BAP}/providers/Microsoft.BusinessAppPlatform/scopes/admin/"
        f"tenantIsolationPolicy?{BAP_VERSION}",
    )
    posture_rows.extend(shape_tenant_isolation(payload))
except Exception as exc:  # noqa: BLE001
    ledger.error("tenantIsolation", exc)

ledger.count("gov_actual_pp_tenant_settings", len(posture_rows))
log("tenant settings", "Created", f"{len(posture_rows)} posture rows")

## Write

In [ ]:
TABLES = [
    ("gov_actual_pp_environments", env_rows),
    ("gov_actual_pp_roles", role_rows),
    ("gov_actual_pp_role_privileges", privilege_rows),
    ("gov_actual_pp_role_assignments", assignment_rows),
    ("gov_actual_pp_resources", resource_rows),
    ("gov_actual_pp_dlp", dlp_rows),
    ("gov_actual_pp_tenant_settings", posture_rows),
]

for table, rows in TABLES:
    write_table(
        spark,  # noqa: F821
        lakehouse_name,
        table,
        stamp(rows, ledger.run_id),
        dry_run=dry_run,
        log=log,
    )

finish(ledger, spark, lakehouse_name, dry_run=dry_run)  # noqa: F821